In [ ]:
import numpy as np

from Datasets.utilities import get_country, country_name_standardizer
from Datasets.wiki_bio.wiki_bio import WikiBio

import os
os.environ['NationalityDatasets'] = "./data/"
bio = WikiBio()

### Reading the raw data

In [ ]:
data = bio.get_raw_data()
data.shape

### Exploring the data

In [ ]:
data.sample(n=10)

In [ ]:
data.nationality.value_counts()

In [ ]:
data.citizenship.value_counts()

In [ ]:
data['birth place'].value_counts()

##### Checking None values

In [ ]:
data.fillna('', inplace=True)

###### checking name

In [ ]:
data[data.name==''].sample(n=10)

In [ ]:
data[data.fullname==''].sample(n=10)

In [ ]:
data[data['birth name']==''].sample(n=10)

In [ ]:
data[(data.name=='') & (data.fullname == '') & (data['birth name'] == '')]['other names'].value_counts()

In [ ]:
print(data.shape)
data = data[~((data.name=='') & (data.fullname=='') & (data['birth name']=='')&(data['other names']==''))]
print(data.shape)

In [ ]:
data.sample(n=10)

In [ ]:
data['full-name'] = data.fullname
data['full-name'] = data.apply(lambda x: x['birth name'] if (x['full-name']=='' or x['full-name'] == 'unknown') else x['full-name'], axis=1)
data['full-name'] = data.apply(lambda x: x['name'] if (x['full-name']=='' or x['full-name'] == 'unknown') else x['full-name'], axis=1)
data['full-name'] = data.apply(lambda x: x['other names'] if (x['full-name']=='' or x['full-name'] == 'unknown') else x['full-name'], axis=1)
data['full-name'].value_counts()

In [ ]:
data[(data['full-name'] == '* -- >')]

In [ ]:
data = data[(data['full-name'] != '* -- >') & (data['full-name'] != '???')]

In [ ]:
data['full-name'].value_counts()

###### checking country

In [ ]:
data[(data['birth place'] == '') & (data['nationality'] == '')  & (data['citizenship'] == '')  & (data['birthplace'] == '')  & (data['home town'] == '')]

In [ ]:
print(data.shape)
data = data[~((data['birth place'] == '') & (data['nationality'] == '')  & (data['citizenship'] == '')  & (data['birthplace'] == '')  & (data['home town'] == ''))]
print(data.shape)

In [ ]:
print(data[(data['birth place'] == '')  & (data['birthplace'] != '')].shape)
data['birth place'] = data.apply(lambda x: x['birthplace'] if x['birth place'] == '' else x['birth place'], axis=1)

In [ ]:
data.shape

###### converting natinality to country

In [ ]:
data[data.nationality=='']

In [ ]:
from Datasets.mapping import nationality_to_country

data['country'] = data.nationality.apply(lambda x: country_name_standardizer(nationality_to_country(x)) if x != '' else '')

###### Extracting birth country

In [ ]:
data['birth_country'] = data['birth place'].apply(lambda x:x.split(',')[-1].strip())

In [ ]:
data['birth_country'] = data['birth_country'].apply(lambda x:x.split()[-2].strip() if (' now ' in x or ' present-day ' in x or ' today ' in x or ' modern ' in x or ' modern-day ' in x or ' present day ' in x or ' later ' in x) else x)

In [ ]:
data['birth_country_2'] = data['birth_country'].apply(lambda x: country_name_standardizer(x))

In [ ]:
data[(data['birth_country'] != '') & (data['birth_country_2'] == '') & (data['country'] == '')].birth_country.value_counts()

In [ ]:
birth_country_mapping = {
    'england': 'United Kingdom',
    'london': 'United Kingdom',
    'west germany': 'Germany',
    'quebec': 'Canada',
    'northern ireland': 'Ireland',
    'new south wales': 'Ireland',
    'wales': 'United Kingdom',
    'scotland': 'United Kingdom',
    'german empire': 'Germany',
    'east germany': 'Germany',
    'berlin': 'Germany',
    'paris': 'France',
    'rome': 'Italy',
    'holy roman empire': 'Italy',
    'andhra pradesh': 'India'
}
data['birth_country_2'] = data.apply(lambda x: birth_country_mapping[x.birth_country] if (x.birth_country_2 == '' and x.birth_country in birth_country_mapping) else x.birth_country_2, axis=1)

In [ ]:
data.country = data.apply(lambda x: x['birth_country_2'] if x.country == '' else x.country, axis=1)

In [ ]:
data.country.value_counts()

In [ ]:
data[data.country == '']

In [ ]:
print(data.shape)
data = data[data.country != '']
print(data.shape)

##### Checking duplicate values

In [ ]:
data['full-name'].value_counts()

In [ ]:
data[data['full-name']=='john smith']

In [ ]:
data[data.duplicated(subset=['full-name', 'country'], keep=False)]

In [ ]:
print(data.shape)
data.drop_duplicates(subset=['full-name', 'country'], keep='first', inplace=True)
print(data.shape)

In [ ]:
data['full-name'].value_counts()[data['full-name'].value_counts()>=2]

In [ ]:
# data.drop_duplicates(subset=['name'], keep=False, inplace=True)

In [ ]:
data.name.value_counts()

In [ ]:
data.shape

### Saving the data

In [ ]:
data['id'] = data.apply(lambda x:'wiki-bio-'+str(x.name), axis=1)


In [ ]:
data.head()

In [ ]:
processed_data = data[['id', 'full-name', 'country']]
processed_data.set_index('id', inplace=True)

In [ ]:
bio.data = processed_data

In [ ]:
bio.save_data(processed_data, 'data.csv')

In [ ]:
bio.save_split_data()